### **EfficientNet in Pytorch**

In [3]:
!pip install efficientnet_pytorch

  Preparing metadata (setup.py) ... done
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl (363.4 MB)
Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl (664.8 MB)
Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl (127.9 MB)
Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl (207.5 MB)
  Created wheel for efficientnet_pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16426 sha256=f9c674bff744acdc14273e417bbe8ba23e91d44582cc04453176003cef6d8d72
  Stored in directory: /root/.cache/pip/wheels/

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from efficientnet_pytorch import EfficientNet

# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# 하이퍼파라미터
batch_size = 64
num_classes = 10
num_epochs = 10
learning_rate = 1e-3

# 데이터 전처리 & 로더
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

train_dataset = datasets.CIFAR10(root='/content/data',
                                 train=True,
                                 download=True,
                                 transform=train_transform)
test_dataset  = datasets.CIFAR10(root='/content/data',
                                 train=False,
                                 download=True,
                                 transform=test_transform)

train_loader = DataLoader(train_dataset,
                          batch_size=batch_size,
                          shuffle=True,
                          num_workers=4)
test_loader = DataLoader(test_dataset,
                         batch_size=batch_size,
                         shuffle=False,
                         num_workers=4)

Using device: cuda


100%|██████████| 170M/170M [00:04<00:00, 35.1MB/s]
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [5]:
# 모델 로드 & 수정
model = EfficientNet.from_pretrained('efficientnet-b0')
model._fc = nn.Linear(model._fc.in_features, num_classes)
model.to(device)

# 손실 & 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 학습
for epoch in range(1, num_epochs+1):
    model.train()
    running_loss = 0.0
    correct = total = 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    print(f'Epoch {epoch}/{num_epochs} ▶ Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.2f}%')

# 7) 테스트 성능
model.eval()
correct = total = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

print(f'\n▶ Test Accuracy: {100. * correct/total:.2f}%')

Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth
100%|██████████| 20.4M/20.4M [00:00<00:00, 237MB/s]


Loaded pretrained weights for efficientnet-b0
Epoch 1/10 ▶ Loss: 0.3858, Acc: 87.15%
Epoch 2/10 ▶ Loss: 0.2225, Acc: 92.48%
Epoch 3/10 ▶ Loss: 0.1736, Acc: 94.16%
Epoch 4/10 ▶ Loss: 0.1498, Acc: 94.92%
Epoch 5/10 ▶ Loss: 0.1304, Acc: 95.57%
Epoch 6/10 ▶ Loss: 0.1142, Acc: 96.07%
Epoch 7/10 ▶ Loss: 0.1021, Acc: 96.40%
Epoch 8/10 ▶ Loss: 0.0939, Acc: 96.72%
Epoch 9/10 ▶ Loss: 0.0913, Acc: 96.87%
Epoch 10/10 ▶ Loss: 0.0816, Acc: 97.21%

▶ Test Accuracy: 93.53%


### 모델 architecture

- `utils.py` : 모델 구성 및 파라미터 로드를 위한 helper 함수들
    - `GlobalParams` & `BlockArgs`: 두 개의 namedtuple
    - `Swish`와 `MemoryEfficientSwish`: 두 가지 Swish 함수 구현
    - `round_filters`와 `round_repeats`: 모델 너비와 깊이 스케일링을 위한 파라미터 계산 함수
    - `get_width_and_height_from_size` 및 `calculate_output_image_size`
    - `drop_connect`: 드롭컨넥트 구현
    - `get_same_padding_conv2d`: `Conv2dDynamicSamePadding`, `Conv2dStaticSamePadding`
    - `get_same_padding_maxPool2d`: `MaxPool2dDynamicSamePadding`, `MaxPool2dStaticSamePadding`
    - EfficientDet 등 다른 모델에서도 사용 가능한 함수

In [6]:
import re
import math
import collections
from functools import partial
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils import model_zoo


################################################################################
# 모델 아키텍처 관련 헬퍼 함수들
################################################################################

# 전체 모델(stem, 모든 블록, head)의 파라미터
GlobalParams = collections.namedtuple('GlobalParams', [
    'width_coefficient', 'depth_coefficient', 'image_size', 'dropout_rate',
    'num_classes', 'batch_norm_momentum', 'batch_norm_epsilon',
    'drop_connect_rate', 'depth_divisor', 'min_depth', 'include_top'])

# 개별 블록의 파라미터
BlockArgs = collections.namedtuple('BlockArgs', [
    'num_repeat', 'kernel_size', 'stride', 'expand_ratio',
    'input_filters', 'output_filters', 'se_ratio', 'id_skip'])

# 기본값 설정
GlobalParams.__new__.__defaults__ = (None,) * len(GlobalParams._fields)
BlockArgs.__new__.__defaults__ = (None,) * len(BlockArgs._fields)

# Swish 활성화 함수
if hasattr(nn, 'SiLU'):
    Swish = nn.SiLU
else:
    # 구버전 PyTorch 호환용
    class Swish(nn.Module):
        def forward(self, x):
            return x * torch.sigmoid(x)


# 메모리 효율적인 Swish 구현
class SwishImplementation(torch.autograd.Function):
    @staticmethod
    def forward(ctx, i):
        result = i * torch.sigmoid(i)
        ctx.save_for_backward(i)
        return result

    @staticmethod
    def backward(ctx, grad_output):
        i = ctx.saved_tensors[0]
        sigmoid_i = torch.sigmoid(i)
        return grad_output * (sigmoid_i * (1 + i * (1 - sigmoid_i)))


class MemoryEfficientSwish(nn.Module):
    def forward(self, x):
        return SwishImplementation.apply(x)


def round_filters(filters, global_params):
    """너비 계수에 따라 필터 수를 계산하고 반올림
       global_params의 width_coefficient, depth_divisor, min_depth 사용

    Args:
        filters (int): 원래 필터 수.
        global_params (namedtuple): 모델의 전역 파라미터

    Returns:
        new_filters: 계산된 새로운 필터 수
    """
    multiplier = global_params.width_coefficient
    if not multiplier:
        return filters
    divisor = global_params.depth_divisor
    min_depth = global_params.min_depth
    filters *= multiplier
    min_depth = min_depth or divisor  # min_depth가 None일 때 divisor 사용
    # 공식 TensorFlow 구현의 수식 따라 구현
    new_filters = max(min_depth, int(filters + divisor / 2) // divisor * divisor)
    if new_filters < 0.9 * filters:  # 10% 이상 과도한 반올림 방지
        new_filters += divisor
    return int(new_filters)


def round_repeats(repeats, global_params):
    """깊이 계수에 따라 블록 반복 수를 계산하고 반올림

    Args:
        repeats (int): 원래 반복 수.
        global_params (namedtuple): 모델의 전역 파라미터

    Returns:
        새로운 반복 수
    """
    multiplier = global_params.depth_coefficient
    if not multiplier:
        return repeats
    # 공식 TensorFlow 구현의 수식 따라 구현
    return int(math.ceil(multiplier * repeats))


def drop_connect(inputs, p, training):
    """Drop connect 구현

    Args:
        inputs (tensor: BxCxHxW): 입력 텐서
        p (float): 드롭 확률 (0~1)
        training (bool): 학습 모드 여부

    Returns:
        output: 드롭컨넥트 적용 후 출력
    """
    assert 0 <= p <= 1, 'p는 [0,1] 범위'

    if not training:
        return inputs

    batch_size = inputs.shape[0]
    keep_prob = 1 - p

    # 확률에 따라 이진 마스크 생성 (0 또는 1)
    random_tensor = keep_prob
    random_tensor += torch.rand([batch_size, 1, 1, 1],
                                dtype=inputs.dtype,
                                device=inputs.device)
    binary_tensor = torch.floor(random_tensor)

    output = inputs / keep_prob * binary_tensor
    return output


def get_width_and_height_from_size(x):
    """입력값으로부터 높이와 너비를 리턴

    Args:
        x (int, tuple or list): 이미지 크기

    Returns:
        size: (H, W) 튜플
    """
    if isinstance(x, int):
        return x, x
    if isinstance(x, (list, tuple)):
        return x
    else:
        raise TypeError()


def calculate_output_image_size(input_image_size, stride):
    """Conv2dSamePadding 사용 시 출력 이미지 크기를 계산
       static padding에 필요

    Args:
        input_image_size (int, tuple or list): 입력 이미지 크기
        stride (int, tuple or list): 스트라이드

    Returns:
        output_image_size: [H, W] 리스트
    """
    if input_image_size is None:
        return None
    image_height, image_width = get_width_and_height_from_size(input_image_size)
    stride = stride if isinstance(stride, int) else stride[0]
    image_height = int(math.ceil(image_height / stride))
    image_width = int(math.ceil(image_width / stride))
    return [image_height, image_width]


# 참고:
# 이하 'SamePadding' 함수는 출력 크기를 ceil(input/stride)로 만듦.
# stride=1일 때만 입력과 같은 크기

def get_same_padding_conv2d(image_size=None):
    """image_size를 지정하면 static padding 사용, 아니면 dynamic padding
       ONNX export 시 static padding 필요

    Args:
        image_size (int or tuple): 이미지 크기

    Returns:
        Conv2dDynamicSamePadding 또는 Conv2dStaticSamePadding
    """
    if image_size is None:
        return Conv2dDynamicSamePadding
    else:
        return partial(Conv2dStaticSamePadding, image_size=image_size)


class Conv2dDynamicSamePadding(nn.Conv2d):
    """TensorFlow의 'SAME' 모드처럼 dynamic 이미지 크기에 대응하는 2D 합성곱
       forward 시 동적으로 padding 계산
    """

    def __init__(self, in_channels, out_channels,
                 kernel_size, stride=1, dilation=1,
                 groups=1, bias=True):
        super().__init__(in_channels, out_channels,
                         kernel_size, stride, 0,
                         dilation, groups, bias)
        self.stride = self.stride if len(self.stride) == 2 \
                      else [self.stride[0]] * 2

    def forward(self, x):
        ih, iw = x.size()[-2:]
        kh, kw = self.weight.size()[-2:]
        sh, sw = self.stride
        oh, ow = math.ceil(ih / sh), math.ceil(iw / sw)
        pad_h = max((oh - 1) * sh + (kh - 1) * self.dilation[0] + 1 - ih, 0)
        pad_w = max((ow - 1) * sw + (kw - 1) * self.dilation[1] + 1 - iw, 0)
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x, [
                pad_w // 2, pad_w - pad_w // 2,
                pad_h // 2, pad_h - pad_h // 2
            ])
        return F.conv2d(x, self.weight, self.bias,
                        self.stride, self.padding,
                        self.dilation, self.groups)


class Conv2dStaticSamePadding(nn.Conv2d):
    """TensorFlow 'SAME' 모드 합성곱, static 이미지 크기에 대응
       __init__에서 padding 계산 후 forward에서 사용
    """

    def __init__(self, in_channels, out_channels,
                 kernel_size, stride=1,
                 image_size=None, **kwargs):
        super().__init__(in_channels, out_channels,
                         kernel_size, stride, **kwargs)
        self.stride = self.stride if len(self.stride) == 2 \
                      else [self.stride[0]] * 2

        # image_size 기반으로 padding 계산 후 저장
        assert image_size is not None
        ih, iw = (image_size, image_size) if isinstance(image_size, int) \
                 else image_size
        kh, kw = self.weight.size()[-2:]
        sh, sw = self.stride
        oh, ow = math.ceil(ih / sh), math.ceil(iw / sw)
        pad_h = max((oh - 1) * sh + (kh - 1) * self.dilation[0] + 1 - ih, 0)
        pad_w = max((ow - 1) * sw + (kw - 1) * self.dilation[1] + 1 - iw, 0)
        if pad_h > 0 or pad_w > 0:
            self.static_padding = nn.ZeroPad2d((
                pad_w // 2, pad_w - pad_w // 2,
                pad_h // 2, pad_h - pad_h // 2
            ))
        else:
            self.static_padding = nn.Identity()

    def forward(self, x):
        x = self.static_padding(x)
        return F.conv2d(x, self.weight, self.bias,
                        self.stride, self.padding,
                        self.dilation, self.groups)


def get_same_padding_maxPool2d(image_size=None):
    """image_size 지정 시 static padding or dynamic padding
       ONNX export 시 static 필요

    Args:
        image_size (int or tuple): 이미지 크기

    Returns:
        MaxPool2dDynamicSamePadding or MaxPool2dStaticSamePadding
    """
    if image_size is None:
        return MaxPool2dDynamicSamePadding
    else:
        return partial(MaxPool2dStaticSamePadding, image_size=image_size)


class MaxPool2dDynamicSamePadding(nn.MaxPool2d):
    """TensorFlow 'SAME' 모드 MaxPooling, dynamic 이미지 크기 대응
       forward 시 padding 계산
    """

    def __init__(self, kernel_size, stride,
                 padding=0, dilation=1,
                 return_indices=False, ceil_mode=False):
        super().__init__(kernel_size, stride,
                         padding, dilation,
                         return_indices, ceil_mode)
        self.stride = [self.stride] * 2 if isinstance(self.stride, int) \
                      else self.stride
        self.kernel_size = [self.kernel_size] * 2 if isinstance(self.kernel_size, int) \
                            else self.kernel_size
        self.dilation = [self.dilation] * 2 if isinstance(self.dilation, int) \
                        else self.dilation

    def forward(self, x):
        ih, iw = x.size()[-2:]
        kh, kw = self.kernel_size
        sh, sw = self.stride
        oh, ow = math.ceil(ih / sh), math.ceil(iw / sw)
        pad_h = max((oh - 1) * sh + (kh - 1) * self.dilation[0] + 1 - ih, 0)
        pad_w = max((ow - 1) * sw + (kw - 1) * self.dilation[1] + 1 - iw, 0)
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x, [
                pad_w // 2, pad_w - pad_w // 2,
                pad_h // 2, pad_h - pad_h // 2
            ])
        return F.max_pool2d(x, self.kernel_size,
                            self.stride, self.padding,
                            self.dilation, self.ceil_mode,
                            self.return_indices)


class MaxPool2dStaticSamePadding(nn.MaxPool2d):
    """TensorFlow 'SAME' 모드 MaxPooling, static 이미지 크기 대응
       __init__에서 padding 계산 후 forward에서 사용
    """

    def __init__(self, kernel_size, stride,
                 image_size=None, **kwargs):
        super().__init__(kernel_size, stride, **kwargs)
        self.stride = [self.stride] * 2 if isinstance(self.stride, int) \
                      else self.stride
        self.kernel_size = [self.kernel_size] * 2 if isinstance(self.kernel_size, int) \
                            else self.kernel_size
        self.dilation = [self.dilation] * 2 if isinstance(self.dilation, int) \
                        else self.dilation

        # image_size 기반 padding 계산 후 저장
        assert image_size is not None
        ih, iw = (image_size, image_size) if isinstance(image_size, int) \
                 else image_size
        kh, kw = self.kernel_size
        sh, sw = self.stride
        oh, ow = math.ceil(ih / sh), math.ceil(iw / sw)
        pad_h = max((oh - 1) * self.stride[0] + (kh - 1) * self.dilation[0] + 1 - ih, 0)
        pad_w = max((ow - 1) * self.stride[1] + (kw - 1) * self.dilation[1] + 1 - iw, 0)
        if pad_h > 0 or pad_w > 0:
            self.static_padding = nn.ZeroPad2d((pad_w // 2, pad_w - pad_w // 2, pad_h // 2, pad_h - pad_h // 2))
        else:
            self.static_padding = nn.Identity()

    def forward(self, x):
        x = self.static_padding(x)
        x = F.max_pool2d(x, self.kernel_size, self.stride, self.padding,
                         self.dilation, self.ceil_mode, self.return_indices)
        return x

- `BlockDecoder`: BlockArgs를 인코딩/디코딩하는 클래스
- `efficientnet_params`: 모델 이름에 대응하는 복합 스케일링 계수 조회 함수
- `get_model_params` 및 `efficientnet`:EfficientNet 모델용 BlockArgs와 GlobalParams를 생성하는 함수들
- `url_map` & `url_map_advprop`: 사전 학습된 가중치 URL 매핑 딕셔너리
- `load_pretrained_weights`: 사전 학습된 가중치를 로드하는 함수

In [7]:
################################################################################
# 모델 파라미터 로드를 위한 헬퍼 함수들
################################################################################

class BlockDecoder(object):
    """가독성을 위한 블록 디코더 클래스
       공식 TensorFlow 저장소에서 그대로 가져옴.
    """

    @staticmethod
    def _decode_block_string(block_string):
        """블록 파라미터 문자열로부터 BlockArgs를 생성

        Args:
            block_string (str): 파라미터를 나타내는 문자열

        Returns:
            BlockArgs: 파일 상단에 정의된 namedtuple
        """
        assert isinstance(block_string, str)

        ops = block_string.split('_')
        options = {}
        for op in ops:
            splits = re.split(r'(\d.*)', op)
            if len(splits) >= 2:
                key, value = splits[:2]
                options[key] = value

        # 스트라이드 검증
        assert (('s' in options and len(options['s']) == 1) or
                (len(options['s']) == 2 and options['s'][0] == options['s'][1]))

        return BlockArgs(
            num_repeat=int(options['r']),
            kernel_size=int(options['k']),
            stride=[int(options['s'][0])],
            expand_ratio=int(options['e']),
            input_filters=int(options['i']),
            output_filters=int(options['o']),
            se_ratio=float(options['se']) if 'se' in options else None,
            id_skip=('noskip' not in block_string))

    @staticmethod
    def _encode_block_string(block):
        """BlockArgs를 문자열 표현으로 인코딩

        Args:
            block (namedtuple): BlockArgs 타입의 namedtuple

        Returns:
            block_string: BlockArgs를 나타내는 문자열
        """
        args = [
            'r%d' % block.num_repeat,
            'k%d' % block.kernel_size,
            's%d%d' % (block.stride[0], block.stride[1]),
            'e%s' % block.expand_ratio,
            'i%d' % block.input_filters,
            'o%d' % block.output_filters
        ]
        if 0 < block.se_ratio <= 1:
            args.append('se%s' % block.se_ratio)
        if block.id_skip is False:
            args.append('noskip')
        return '_'.join(args)

    @staticmethod
    def decode(string_list):
        """네트워크 내 블록을 지정하는 문자열 목록을 디코딩

        Args:
            string_list (list[str]): 블록 파라미터 문자열 리스트

        Returns:
            blocks_args: BlockArgs namedtuple 리스트
        """
        assert isinstance(string_list, list)
        blocks_args = []
        for block_string in string_list:
            blocks_args.append(BlockDecoder._decode_block_string(block_string))
        return blocks_args

    @staticmethod
    def encode(blocks_args):
        """BlockArgs 목록을 문자열 목록으로 인코딩

        Args:
            blocks_args (list[namedtuples]): BlockArgs namedtuple 리스트

        Returns:
            block_strings: 블록 파라미터 문자열 리스트
        """
        block_strings = []
        for block in blocks_args:
            block_strings.append(BlockDecoder._encode_block_string(block))
        return block_strings


def efficientnet_params(model_name):
    """EfficientNet 모델 이름을 복합 스케일링 계수로 매핑

    Args:
        model_name (str): 조회할 모델 이름

    Returns:
        (width, depth, resolution, dropout) 튜플
    """
    params_dict = {
        # 계수: 너비, 깊이, 해상도, 드롭아웃 비율
        'efficientnet-b0': (1.0, 1.0, 224, 0.2),
        'efficientnet-b1': (1.0, 1.1, 240, 0.2),
        'efficientnet-b2': (1.1, 1.2, 260, 0.3),
        'efficientnet-b3': (1.2, 1.4, 300, 0.3),
        'efficientnet-b4': (1.4, 1.8, 380, 0.4),
        'efficientnet-b5': (1.6, 2.2, 456, 0.4),
        'efficientnet-b6': (1.8, 2.6, 528, 0.5),
        'efficientnet-b7': (2.0, 3.1, 600, 0.5),
        'efficientnet-b8': (2.2, 3.6, 672, 0.5),
        'efficientnet-l2': (4.3, 5.3, 800, 0.5),
    }
    return params_dict[model_name]


def efficientnet(width_coefficient=None, depth_coefficient=None, image_size=None,
                 dropout_rate=0.2, drop_connect_rate=0.2, num_classes=1000, include_top=True):
    """EfficientNet 모델의 BlockArgs와 GlobalParams 생성

    Args:
        width_coefficient (float): 너비 계수
        depth_coefficient (float): 깊이 계수
        image_size (int): 입력 해상도
        dropout_rate (float): 드롭아웃 비율
        drop_connect_rate (float): 드롭컨넥트 비율
        num_classes (int): 클래스 수
        include_top (bool): 마지막 FC 레이어 포함 여부

    Returns:
        blocks_args, global_params: 네트워크 구성 파라미터
    """

    # 전체 모델(b0 기준) 블록 파라미터 목록
    # EfficientNet 클래스 생성 시 모델에 맞게 수정됨
    blocks_args = [
        'r1_k3_s11_e1_i32_o16_se0.25',
        'r2_k3_s22_e6_i16_o24_se0.25',
        'r2_k5_s22_e6_i24_o40_se0.25',
        'r3_k3_s22_e6_i40_o80_se0.25',
        'r3_k5_s11_e6_i80_o112_se0.25',
        'r4_k5_s22_e6_i112_o192_se0.25',
        'r1_k3_s11_e6_i192_o320_se0.25',
    ]
    blocks_args = BlockDecoder.decode(blocks_args)

    global_params = GlobalParams(
        width_coefficient=width_coefficient,
        depth_coefficient=depth_coefficient,
        image_size=image_size,
        dropout_rate=dropout_rate,

        num_classes=num_classes,
        batch_norm_momentum=0.99,
        batch_norm_epsilon=1e-3,
        drop_connect_rate=drop_connect_rate,
        depth_divisor=8,
        min_depth=None,
        include_top=include_top,
    )

    return blocks_args, global_params


def get_model_params(model_name, override_params):
    """모델 이름에 대한 블록 파라미터와 글로벌 파라미터를 가져옴.

    Args:
        model_name (str): 모델 이름
        override_params (dict): global_params 덮어쓰기용 딕셔너리

    Returns:
        blocks_args, global_params
    """
    if model_name.startswith('efficientnet'):
        w, d, s, p = efficientnet_params(model_name)
        # 모든 모델의 drop_connect_rate는 0.2로 고정
        blocks_args, global_params = efficientnet(
            width_coefficient=w, depth_coefficient=d,
            dropout_rate=p, image_size=s)
    else:
        raise NotImplementedError(f'정의되지 않은 모델 이름: {model_name}')
    if override_params:
        # 잘못된 키가 있으면 ValueError 발생
        global_params = global_params._replace(**override_params)
    return blocks_args, global_params


# Standard 학습용 URL 매핑
# 논문: EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks
url_map = {
    'efficientnet-b0': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth',
    'efficientnet-b1': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b1-f1951068.pth',
    'efficientnet-b2': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b2-8bb594d6.pth',
    'efficientnet-b3': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b3-5fb5a3c3.pth',
    'efficientnet-b4': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b4-6ed6700e.pth',
    'efficientnet-b5': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b5-b6417697.pth',
    'efficientnet-b6': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b6-c76e70fd.pth',
    'efficientnet-b7': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b7-dcc49843.pth',
}

# AdvProp(Adversarial Examples) 학습용 URL 매핑
# 논문: Adversarial Examples Improve Image Recognition
url_map_advprop = {
    'efficientnet-b0': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/adv-efficientnet-b0-b64d5a18.pth',
    'efficientnet-b1': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/adv-efficientnet-b1-0f3ce85a.pth',
    'efficientnet-b2': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/adv-efficientnet-b2-6e9d97e5.pth',
    'efficientnet-b3': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/adv-efficientnet-b3-cdd7c0f4.pth',
    'efficientnet-b4': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/adv-efficientnet-b4-44fb3a87.pth',
    'efficientnet-b5': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/adv-efficientnet-b5-86493f6b.pth',
    'efficientnet-b6': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/adv-efficientnet-b6-ac80338e.pth',
    'efficientnet-b7': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/adv-efficientnet-b7-4652b6dd.pth',
    'efficientnet-b8': 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/adv-efficientnet-b8-22a8fe65.pth',
}


def load_pretrained_weights(model, model_name, weights_path=None, load_fc=True, advprop=False, verbose=True):
    """사전 학습 가중치를 로컬 파일 또는 URL에서 로드

    Args:
        model (Module): 전체 EfficientNet 모델 인스턴스
        model_name (str): 모델 이름
        weights_path (None or str):
            str: 로컬 디스크상의 가중치 파일 경로
            None: 인터넷에서 가중치 다운로드
        load_fc (bool): 마지막 FC 레이어 가중치 로드 여부
        advprop (bool): AdvProp 가중치 로드시 True
        verbose (bool): 로드 완료 메시지 출력 여부
    """
    if isinstance(weights_path, str):
        state_dict = torch.load(weights_path)
    else:
        # AutoAugment용 또는 AdvProp용 URL 선택
        url_map_ = url_map_advprop if advprop else url_map
        state_dict = model_zoo.load_url(url_map_[model_name])

    if load_fc:
        ret = model.load_state_dict(state_dict, strict=False)
        assert not ret.missing_keys, f'가중치 로드 실패(누락 키): {ret.missing_keys}'
    else:
        # FC 레이어만 제외하고 로드
        state_dict.pop('_fc.weight')
        state_dict.pop('_fc.bias')
        ret = model.load_state_dict(state_dict, strict=False)
        assert set(ret.missing_keys) == {'_fc.weight', '_fc.bias'}, \
            f'가중치 로드 실패(누락 키): {ret.missing_keys}'
    assert not ret.unexpected_keys, f'가중치 로드 실패(예상치 못한 키): {ret.unexpected_keys}'

    if verbose:
        print(f'{model_name} 사전 학습 가중치 로드 완료')